In [1]:
import openai, json, requests

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [3]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return response.json()

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    response.raise_for_status()
    return response.json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [6]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the current popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a movie by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get similar movies by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [7]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()

    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )

    process_ai_response(response.choices[0].message)

In [8]:
while True:
    message = input("Send a message to the Movie Agent...")

    if message == "quit" or message == "q":
        break

    messages.append({"role": "user", "content": message})
    print(f"User: {message}")

    call_ai()

User: 지금 인기 있는 영화 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {}
Agent: 현재 인기 있는 영화들을 소개할게요:

1. **Obsession**
   - **개요**: "One Wish Willow"를 깨뜨린 후, 한 로맨틱한 남자가 그의 사랑을 얻기 위해 노력하지만, 모든 욕망에는 어두운 대가가 따르는 것을 알게 된다.
   - **개봉일**: 2026-05-13
   - **평점**: 7.9
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - **개요**: 1980년대 안드라 프라데시의 시골 마을에서 한 열정적인 주민이 스포츠를 통해 그의 커뮤니티를 하나로 묶고 강력한 적에 맞선다.
   - **개봉일**: 2026-06-03
   - **평점**: 6.8
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **The Unknown Man**
   - **개요**: 벨기에 작가 루이즈가 코트다쥐르에서 고립되기를 결정하며 영감을 찾으려는 과정을 담고 있다.
   - **개봉일**: 2021-10-16
   - **평점**: 8.1
   - ![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

4. **Hai Jawani Toh Ishq Hona Hai**
   - **개요**: 결혼을 두고 우선순위가 충돌한 Jass는 해외에서 새로운 로맨스에 불행한 사실들이 겹치며 진정한 사랑의 의미를 발견하게 된다.
   - **개봉일**: 2026-06-04
   - **평점**: 5.4
   - ![Hai Jawani Toh Is